In [2]:
import gymnasium as gym
import flappy_bird_gymnasium 
import torch 
import torch.nn as nn 
from collections import deque # for making Replay memory
import random
import itertools # A module for looping indefinetly 
import yaml 
import torch.optim as optim 

In [ ]:
# Does initial device check for better computation
if torch.backend.mps.is_available(): 
    device= "mps"
elif torch.cuda.is_available(): 
    device= "cuda"
else: 
    device= "cpu"

In [ ]:
# creating a replay memory  
class Replaymemory(): 
    # Creating FIFO memory
    def __init__(self, maxlen, seed= None): # maxlen tells how nany experience will memory store, seed makes the random choice to be in same order every-time 
        self.memory= deque([], maxlen=maxlen) # A deque of empyt array with maxlen 

    def append(self, new_exp): 
        self.memory.append(new_exp) # will append experiencec into memory
        
    def sample(self, sample_size):
        return random.sample(self.memory, sample_size) # will return a batch of sample size

    # Private func. to find buffer size
    def __len__(self):
        return len(self.memory)

In [ ]:
# Since policy and terget network has same nn architecture we are making it a class to use it multiple times.
class DQN(nn.Module): 
    # i/p, o/p & hidden dimensions 
    def __init__(self, state_dim=12, action_dim=2, hidden_dim=256): # Given default values
        super(DQN, self).__init__()

        self.model= nn.Sequential(
            nn.Linear(state_dim,hidden_dim), 
            nn.ReLU(), 
            nn.Linear(hidden_dim, action_dim)
        )
    def forward(self, x): 
        return self.model(x)

In [11]:
# A function to test and train 
class agent():
    # importing parametrs 
    
    def __init__(self, parameter_set): # parameter set will take a vlaue
        self.parameter_set= parameter_set
        with open("parameter.yaml", "r") as f: 
            all_param_set= yaml.safe_load(f) # here all values from yaml file is converted into python dictionary.
            params= all_param_set[parameter_set] # params will have values which comes under parameter_set value from the "f" opened file

        self.alpha= params["alpha"] 
        self.gamma= params["gamma"]
        self.epsilon_init= params["epsilon_init"] 
        
        self.epsilon_min= params["epsilon_min"] 
        self.episode_decay= params["epsilon_decay"] 
        self.replay_memory_size= params["replay_memory_size"]
        
        self.mini_batch_size= params["mini_batch_size"] 
        self.reward_threshold= params["reward_threshold"] 
        self.network_sync_rate= params["network_sync_rate"]

        #Loss funtion for nn 
        self.loss_fn= nn.MSELoss() 
        self.optimizer= None
    
    def run(self, is_training= True, render= False): # Giving default values, so it only renders when its testing
        
        
        env = gym.make("FlappyBird-v0", render_mode="human" if render else None ,use_lidar=True) # rendering only when render = true 
        num_states= env.obervation_space.shape[0] # i/p dimensions
        num_actions= env.action_space.n # o/p dim.
        
        policy_dqn= DQN(num_states, num_actions).to(device) # Created policy dqn and passed it in the found device
       
        if is_training: 
            memory = ReplayMemory(self.replay_memory_size)
            epsilon= self.epsilon_init
            # target network 
            target_dqn= DQN(num_states, num_actions).to(device) 
            # Loading a copy of the policy network weights and bias 
            target_dqn.load_state_dict(policy_dqn.state_dict()) 

            steps= 0 
            self.optimizer= optim.Adam(policy_dqn.parameters(), lr= self.alpha) # creating an optimizer for policy net. only when training
        
        for episode in itertools.count(): # For all the episodes
            state,_ = env.reset() # resetting env. for each episodes, and giving initial state and some info.
            state= torch.tensor(state, dtype= torch.float, device= device) # since state value cannot be processed by pytorch so we will convert them into tensors.
            
            episode_reward=0 
            terminated= False 
            
            while not terminated: # For each Episode
                # E-greedy policy
                if is_training and random.random< epsilon: 
                    action = env.action_space.sample() # taking action to explore
                    action = torch.tensor(action, dtype= torch.long, device = device) # Only in training to feed the nn.
                else:
                    with torch.no_grad(): # to stop gradient calculation in testing
                        action = policy_dqn(state.unsqueeze(dim=0)).squeeze().argmax() # taking action to exploit ,argmax to take the max. value(Q-learning)
                
                next_state, reward, terminated,_,_= env.step(action.item()) # Implimenting the action(since tensor valu so using item)
                # tensors 
                next_state= torch.tensor(next_state, dtype= torch.float, device= device) 
                reward= torch.tensor(reward, dtype= torch.float, device= device) 
                
                if is_training: 
                    memory.append((state, action,new_state, reward, terminated)) # these are the vales of each new experience, added to replay memory if training.
                    steps+= 1
    
                state= new_state 
                episode_reward+= reward 
            print(f"For episode= {episode+1}, total reward= {episode_reward} & epsilon = {epsilon}")

            if is_training:
                # E-decay at each episode 
                epsilon= max(epsilon*self.episode_decay, self.epsilon_min) # this will ensure "E" decays till min value.

            # syncing the 2 networks when learning and no. of experiences in replay mem. > mini_batch_size
            if is_training and len(memory) > self.mini_batch_size:  
                # get samples 
                mini_batch= memory.sample(self.mini_batch_size) 
                optimize(mini_batch, policy_dqn, target_dqn) 

                # sync networks 
                if steps > self.network_sync_rate: 
                    target_dqn.load_state_dict(policy_dqn.state_dict())
                    step= 0 # reseting to 0 so count till 10 again
        # env.close() ,not using this, since we want to stop the code manually
        
    def optimize(self, mini_batch, policy_dqn, target_dqn): 
        # get experience 
        for state, action, next_state, reward, terminated in mini_batch: 
            if terminated: 
                traget= reward 
            else:
                with torch.no_grad(): 
                    target_q= reward + self.gamma*target_dqn(next_state).max() # this is the target Q-value or y-true value
            current_q = policy_dqn(state) 

            #loss 
            loss= self.loss_fn(current_q, target_q)  
            self.optimizer.zero_grad() # Making gradient 0 ,so it do not accumulate
            loss.backward() # finding loss 
            self.optimizer.step() # optimizing weights

        def optimize(self, mini_batch, policy_dqn, target_dqn):
        # get batch of experiences
        states, actions, next_states, rewards, terminations = zip(*mini_batch)

        states = torch.stack(states)
        actions = torch.stack(actions)
        next_states = torch.stack(next_states)
        rewards = torch.stack(rewards)
        terminations = torch.tensor(terminations).float().to(device)

        # calculate target Q-values - if terminations=true => zero
        with torch.no_grad():
            target_q = rewards + (1-terminations) * self.gamma * target_dqn(next_states).max(dim=1)[0]

            
        # calculate y_pred i.e. Q-value from current policy
        current_q = policy_dqn(states).gather(dim=1, index=actions.unsqueeze(dim=1)).squeeze()

        # compute loss
        loss = self.loss_fn(current_q, target_q)

        # optimize model
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()